In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
import timm
from sklearn.metrics import precision_score, recall_score, f1_score
import time
import numpy as np

# ===============================
# CONFIG
# ===============================
DATA_DIR = r"C:\Users\m8494\Downloads\7030data"   # your dataset folder
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
VAL_DIR = os.path.join(DATA_DIR, 'val')

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 80
LR = 1e-4
NUM_CLASSES = 3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ===============================
# TRANSFORMS
# ===============================
train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5])
])

val_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5])
])

# ===============================
# LOAD DATA
# ===============================
train_data = datasets.ImageFolder(TRAIN_DIR, transform=train_tfms)
val_data = datasets.ImageFolder(VAL_DIR, transform=val_tfms)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False)

print("Classes:", train_data.classes)
print("Train:", len(train_data), " Val:", len(val_data))
print("Device:", DEVICE)

# ===============================
# MODEL (ViT-B/16)
# ===============================
model = timm.create_model("vit_base_patch16_224", pretrained=True)
model.head = nn.Linear(model.head.in_features, NUM_CLASSES)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

# ===============================
# METRIC STORAGE
# ===============================
train_loss_hist = []
val_loss_hist = []
train_acc_hist = []
val_acc_hist = []
prec_hist = []
recall_hist = []
f1_hist = []
epoch_time_hist = []

# ===============================
# TRAINING LOOP
# ===============================
total_start = time.time()
best_val_acc = 0.0

for epoch in range(1, EPOCHS+1):

    epoch_start = time.time()

    model.train()
    running_loss = 0
    correct = 0
    total = 0

    # ---------------- TRAIN ----------------
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc = correct / total

    # ---------------- VAL ----------------
    model.eval()
    val_running_loss = 0
    val_correct = 0
    val_total = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)

            loss = criterion(outputs, labels)
            val_running_loss += loss.item()

            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_loss = val_running_loss / len(val_loader)
    val_acc = val_correct / val_total

    # METRICS
    precision = precision_score(all_labels, all_preds, average='macro')
    recall = recall_score(all_labels, all_preds, average='macro')
    f1 = f1_score(all_labels, all_preds, average='macro')

    epoch_time = time.time() - epoch_start

    # STORE METRICS
    train_loss_hist.append(train_loss)
    val_loss_hist.append(val_loss)
    train_acc_hist.append(train_acc)
    val_acc_hist.append(val_acc)
    prec_hist.append(precision)
    recall_hist.append(recall)
    f1_hist.append(f1)
    epoch_time_hist.append(epoch_time)

    # PRINT
    print(f"Epoch {epoch}/{EPOCHS} → "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc*100:.2f}% | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc*100:.2f}% | "
          f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f} | "
          f"Time: {epoch_time:.2f}s")

    # SAVE BEST MODEL
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "vit_best_model.pth")
        print("Saved Best Model")

total_end = time.time()

# ===============================
# AVERAGE METRICS
# ===============================
print("\n=========== AVERAGE METRICS ===========")
print(f"Avg Train Loss: {np.mean(train_loss_hist):.4f}")
print(f"Avg Val Loss: {np.mean(val_loss_hist):.4f}")
print(f"Avg Train Accuracy: {np.mean(train_acc_hist)*100:.2f}%")
print(f"Avg Val Accuracy: {np.mean(val_acc_hist)*100:.2f}%")
print(f"Avg Precision: {np.mean(prec_hist):.4f}")
print(f"Avg Recall: {np.mean(recall_hist)::.4f}")
print(f"Avg F1-score: {np.mean(f1_hist):.4f}")
print(f"Avg Time/Epoch: {np.mean(epoch_time_hist):.2f}s")
print(f"Total Training Time: {(total_end-total_start):.2f}s ({(total_end-total_start)/60:.2f} min)")
print("========================================")

Classes: ['grade1', 'grade2', 'grade3']
Train: 420  Val: 180
Device: cpu
Epoch 1/80 → Train Loss: 1.2097, Train Acc: 45.00% | Val Loss: 0.9912, Val Acc: 56.11% | Precision: 0.6792, Recall: 0.5611, F1: 0.4808 | Time: 327.75s
Saved Best Model
Epoch 2/80 → Train Loss: 0.8063, Train Acc: 60.00% | Val Loss: 0.7751, Val Acc: 61.11% | Precision: 0.6592, Recall: 0.6111, F1: 0.5134 | Time: 327.70s
Saved Best Model
Epoch 3/80 → Train Loss: 0.6259, Train Acc: 70.71% | Val Loss: 0.6520, Val Acc: 71.67% | Precision: 0.7099, Recall: 0.7167, F1: 0.6941 | Time: 316.45s
Saved Best Model
Epoch 4/80 → Train Loss: 0.4774, Train Acc: 80.24% | Val Loss: 0.7518, Val Acc: 67.78% | Precision: 0.7028, Recall: 0.6778, F1: 0.6790 | Time: 315.17s
Epoch 5/80 → Train Loss: 0.4420, Train Acc: 83.10% | Val Loss: 0.7613, Val Acc: 69.44% | Precision: 0.6856, Recall: 0.6944, F1: 0.6663 | Time: 316.02s
Epoch 6/80 → Train Loss: 0.3712, Train Acc: 84.76% | Val Loss: 0.7071, Val Acc: 68.89% | Precision: 0.7018, Recall: 0.688

ValueError: Invalid format specifier ':.4f' for object of type 'float'